# Wahla — Financial Digital Twin (full backend, inline)

> A dashboard tells you what you spent; a chatbot tells you what your data says;
> **our Twin tells you who you will be in 24 months if you make this decision** —
> and shows exactly which parts of your financial self change and why.

Every module of the backend — cleaning, feature engineering, the Twin engine,
memory, simulation, and the live API — is defined **directly in the cells
below** as real, editable, runnable code (`%%writefile` writes each cell to its
real path so imports work, then later cells import and run the functions
directly — nothing here is a hidden file or a black-box `!python3 script.py`
call). Only the raw dataset itself is fetched, since it's data, not code.

The React frontend is not included — it's a browser UI, not something Colab
renders; everything it talks to (the API) is fully live in this notebook.

Repo: https://github.com/AbdullahAlbadri/Wahla


## 0. Setup — install deps, create the folder layout, fetch *only the data*

In [ ]:
!pip install -q fastapi uvicorn pydantic pandas numpy nest-asyncio requests


In [ ]:
!mkdir -p twin data/raw data/processed


In [ ]:
import urllib.request

RAW = "https://raw.githubusercontent.com/AbdullahAlbadri/Wahla/main/data/raw/"
DATA_FILES = ["fin_account.tsv", "fin_client.tsv", "fin_disp.tsv", "fin_order.tsv",
              "fin_trans.tsv", "fin_loan.tsv", "fin_card.tsv", "fin_district.tsv"]

for fname in DATA_FILES:
    dest = f"data/raw/{fname}"
    urllib.request.urlretrieve(RAW + fname, dest)
    print(dest)


## 1. `twin/` — the isolated Twin package

Every file below is the literal source from the repo, written to disk exactly
as-is so it can be imported normally. Edit any cell and re-run it (then
re-import) to change real behavior.


### `twin/__init__.py`

In [ ]:
%%writefile twin/__init__.py
"""Financial Digital Twin package."""


### `twin/config.py` — central configuration, no hardcoded values scattered in modules

In [ ]:
%%writefile twin/config.py
"""Central configuration — no hardcoded values scattered in modules.

Swap DATA_DIR / loader to point at a real Open Banking API later:
only `data_loader.py` needs to change, everything downstream is schema-driven.
"""
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "data" / "processed"

# ---- Berka source files (tab-delimited, no headers) ----
FILES = {
    "account": "fin_account.tsv",
    "client": "fin_client.tsv",
    "disp": "fin_disp.tsv",
    "order": "fin_order.tsv",
    "trans": "fin_trans.tsv",
    "loan": "fin_loan.tsv",
    "card": "fin_card.tsv",
    "district": "fin_district.tsv",
}

COLUMNS = {
    "account": ["account_id", "district_id", "create_date", "frequency"],
    "client": ["client_id", "birth_date", "gender", "district_id"],
    "disp": ["disp_id", "client_id", "account_id", "disp_type"],
    "order": ["order_id", "account_id", "bank_to", "account_to", "amount", "category"],
    "trans": ["trans_id", "account_id", "trans_date", "amount", "balance",
              "trans_type", "operation", "category", "other_bank_id", "other_account_id"],
    "loan": ["loan_id", "account_id", "granted_date", "amount", "duration", "payments", "status"],
    "card": ["card_id", "disp_id", "card_type", "issued_date"],
    "district": ["district_id", "district_name", "region", "population",
                 "n_muni_lt_500", "n_muni_500_2k", "n_muni_2k_10k", "n_muni_gt_10k",
                 "n_cities", "urban_ratio", "avg_salary", "unemp_95", "unemp_96",
                 "entrepreneurs_per_1k", "crimes_95", "crimes_96"],
}

# ---- Open-Banking-style semantic mappings (normalize bank codes → categories) ----
TRANS_TYPE = {"C": "credit", "D": "debit", "P": "cash_withdrawal"}
OPERATION = {
    "CCW": "card_withdrawal",
    "CIC": "cash_deposit",
    "COB": "incoming_transfer",
    "WIC": "cash_withdrawal",
    "ROB": "outgoing_transfer",
}
CATEGORY = {
    "IC": "interest_income",
    "IO": "overdraft_fee",
    "PE": "pension_income",
    "LO": "loan_payment",
    "HH": "household",
    "ST": "bank_fee",
    "IN": "insurance",
    "SA": "salary",           # derived: large recurring incoming credit
    "UN": "uncategorized",
}
ORDER_CATEGORY = {"HH": "household", "IN": "insurance", "LO": "loan_payment", "LE": "leasing"}
LOAN_STATUS = {"A": "finished_ok", "B": "defaulted", "C": "running_ok", "D": "running_in_debt"}
CARD_TYPE = {"J": "junior", "C": "classic", "G": "gold"}

# ---- Feature engineering parameters ----
SALARY_MIN_AMOUNT = 300          # incoming credit above this, recurring monthly → salary
SALARY_MONTHLY_TOLERANCE_DAYS = 6
RECURRING_MIN_OCCURRENCES = 3
EMERGENCY_FUND_MONTHS_TARGET = 6  # standard financial planning benchmark
HEALTH_SCORE_WEIGHTS = {
    "savings_rate": 0.30,
    "cashflow_stability": 0.20,
    "debt_ratio": 0.25,
    "emergency_fund": 0.25,
}


### `twin/data_loader.py` — the ONLY module allowed to touch raw data

Cleans the Berka batch export *and* a live Open Banking-style feed through the
same `_finalize_transactions()` / `_finalize_loans()` choke points — type
coercion, dedup, dropping rows that would otherwise crash a downstream
`float()`/`round()` call, mixed-date-format handling.

In [ ]:
%%writefile twin/data_loader.py
"""Data ingestion + cleaning layer.

This is the ONLY module that knows about the raw dataset format.
Replacing Berka with a real Open Banking API = reimplement `load_all()`
to return the same normalized frames.
"""
import pandas as pd

from . import config


def _read(table: str) -> pd.DataFrame:
    path = config.DATA_DIR / config.FILES[table]
    df = pd.read_csv(path, sep="\t", header=None, names=config.COLUMNS[table],
                     dtype=str, keep_default_na=False, na_values=[""])
    return df


def _finalize_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Shared cleaning choke point for every ingestion path (batch TSV or live API).

    Strict typing, drops invalid rows, dedupes, derives signed/absolute amount
    and month. Callers must normalize trans_type/operation/category into
    open-banking labels *before* reaching here — see load_transactions() for
    the Berka-code mapping and load_live_transactions() for a live feed.
    check_architecture.py enforces that nothing downstream (features.py,
    engine.py, ...) ever imports data_loader or touches a raw file directly,
    so this is the only place cleaning can happen.
    """
    # format="mixed": a live feed mixes date-only and full-timestamp rows —
    # pandas otherwise infers one format from the first row and silently
    # NaTs (then drops) every row that doesn't match it
    df["trans_date"] = pd.to_datetime(df["trans_date"], format="mixed", errors="coerce")
    for col in ("amount", "balance"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["account_id"] = df["account_id"].astype(int)

    df = df.dropna(subset=["trans_date", "amount"])
    df = df.drop_duplicates(subset=["trans_id"])

    # this Berka variant stores amounts already signed (debits negative);
    # keep signed_amount as-is and expose magnitude separately
    df["signed_amount"] = df["amount"]
    df["amount"] = df["amount"].abs()
    df["month"] = df["trans_date"].dt.to_period("M")
    return df


def load_transactions() -> pd.DataFrame:
    """Load, clean and normalize the transactions table from the Berka TSV export."""
    df = _read("trans")
    df["trans_type"] = df["trans_type"].str.strip().map(config.TRANS_TYPE).fillna("unknown")
    df["operation"] = df["operation"].str.strip().map(config.OPERATION).fillna("other")
    df["category"] = df["category"].str.strip().map(config.CATEGORY).fillna("uncategorized")
    return _finalize_transactions(df)


LIVE_REQUIRED_FIELDS = {"trans_id", "account_id", "trans_date", "amount", "trans_type"}


def load_live_transactions(raw: list[dict]) -> pd.DataFrame:
    """Clean + normalize a batch of transactions pulled live from an Open Banking API.

    This is the connect-time equivalent of load_transactions(): a real bank
    feed lands here first and is forced through the exact same typing/dedup/
    derivation contract as the Berka batch pipeline — only the cleaned frame
    is allowed to reach features.py. Expected shape per transaction:
        {trans_id, account_id, trans_date (ISO 8601), amount,
         trans_type ("credit"|"debit"|"cash_withdrawal"),
         balance?, operation?, category?}
    """
    columns = ["trans_id", "account_id", "trans_date", "amount", "balance",
               "trans_type", "operation", "category", "signed_amount", "month"]
    if not raw:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(raw)
    missing = LIVE_REQUIRED_FIELDS - set(df.columns)
    if missing:
        raise ValueError(f"live transaction feed missing required fields: {sorted(missing)}")

    for col in ("balance", "operation", "category"):
        if col not in df.columns:
            df[col] = None
    df["operation"] = df["operation"].fillna("other").astype(str).str.strip()
    df["category"] = df["category"].fillna("uncategorized").astype(str).str.strip()
    df["trans_type"] = df["trans_type"].astype(str).str.strip().str.lower()

    return _finalize_transactions(df)


def load_accounts() -> pd.DataFrame:
    df = _read("account")
    df["account_id"] = df["account_id"].astype(int)
    df["district_id"] = df["district_id"].astype(int)
    df["create_date"] = pd.to_datetime(df["create_date"], errors="coerce")
    return df


def load_clients() -> pd.DataFrame:
    df = _read("client")
    df["client_id"] = df["client_id"].astype(int)
    df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")
    df["district_id"] = pd.to_numeric(df["district_id"], errors="coerce")
    return df


def load_dispositions() -> pd.DataFrame:
    df = _read("disp")
    for col in ("disp_id", "client_id", "account_id"):
        df[col] = df[col].astype(int)
    return df


def _finalize_loans(df: pd.DataFrame) -> pd.DataFrame:
    """Shared cleaning choke point for loan records (batch TSV or live API).

    memory.py and features.py use loan_id/account_id/amount/duration/payments
    unconditionally (float()/round()/f-string formatting, no null-guards), so
    unlike granted_date this function drops any row missing them instead of
    letting NaN slip through to a crash later. Callers must normalize `status`
    into an open-banking label before reaching here — see load_loans() for
    the Berka-code mapping and load_live_loans() for a live feed.
    """
    df["granted_date"] = pd.to_datetime(df["granted_date"], format="mixed", errors="coerce")
    for col in ("amount", "duration", "payments"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["loan_id"] = pd.to_numeric(df["loan_id"], errors="coerce")
    df["account_id"] = pd.to_numeric(df["account_id"], errors="coerce")

    df = df.dropna(subset=["loan_id", "account_id", "amount", "duration", "payments"])
    df["loan_id"] = df["loan_id"].astype(int)
    df["account_id"] = df["account_id"].astype(int)
    df = df.drop_duplicates(subset=["loan_id"])
    return df


def load_loans() -> pd.DataFrame:
    """Load, clean and normalize the loans table from the Berka TSV export."""
    df = _read("loan")
    df["status"] = df["status"].str.strip().map(config.LOAN_STATUS).fillna("unknown")
    return _finalize_loans(df)


LIVE_LOAN_REQUIRED_FIELDS = {"loan_id", "account_id", "amount", "status"}


def load_live_loans(raw: list[dict]) -> pd.DataFrame:
    """Clean + normalize loan records pulled live from an Open Banking API.

    Connect-time equivalent of load_loans(): mirrors load_live_transactions()
    — forces a real feed through the same typing/dedup/dropna contract as the
    Berka batch loans table. `status` is expected already human-readable
    (not a Berka code), e.g. "running_ok" | "running_in_debt" |
    "finished_ok" | "defaulted".
    """
    columns = ["loan_id", "account_id", "granted_date", "amount",
               "duration", "payments", "status"]
    if not raw:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(raw)
    missing = LIVE_LOAN_REQUIRED_FIELDS - set(df.columns)
    if missing:
        raise ValueError(f"live loan feed missing required fields: {sorted(missing)}")

    for col in ("granted_date", "duration", "payments"):
        if col not in df.columns:
            df[col] = None
    df["status"] = df["status"].fillna("unknown").astype(str).str.strip().str.lower()

    return _finalize_loans(df)


def load_orders() -> pd.DataFrame:
    df = _read("order")
    df["order_id"] = df["order_id"].astype(int)
    df["account_id"] = df["account_id"].astype(int)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["category"] = df["category"].str.strip().map(config.ORDER_CATEGORY).fillna("other")
    return df


def load_cards() -> pd.DataFrame:
    df = _read("card")
    df["card_id"] = df["card_id"].astype(int)
    df["disp_id"] = df["disp_id"].astype(int)
    df["card_type"] = df["card_type"].str.strip().map(config.CARD_TYPE).fillna("unknown")
    df["issued_date"] = pd.to_datetime(df["issued_date"], errors="coerce")
    return df


def load_districts() -> pd.DataFrame:
    df = _read("district")
    df["district_id"] = df["district_id"].astype(int)
    for col in ("population", "avg_salary", "urban_ratio", "unemp_96"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df[["district_id", "district_name", "region", "population",
               "urban_ratio", "avg_salary", "unemp_96"]]


def load_all() -> dict:
    """Load every table, cleaned and normalized."""
    return {
        "transactions": load_transactions(),
        "accounts": load_accounts(),
        "clients": load_clients(),
        "dispositions": load_dispositions(),
        "loans": load_loans(),
        "orders": load_orders(),
        "cards": load_cards(),
        "districts": load_districts(),
    }


### `twin/personality.py` — financial personality inference (transparent formulas, not hardcoded labels)

In [ ]:
%%writefile twin/personality.py
"""Financial Personality inference.

Each personality gets a 0..1 score computed from engineered Twin features
(never hardcoded labels). The winner is the highest score; confidence is
the winner's margin over the runner-up, normalized:

    confidence = top / (top + second)      # 0.5 = tie, 1.0 = unambiguous

Scores are transparent formulas over documented features, so every
classification is explainable attribute-by-attribute.
"""
import numpy as np


def _clip(x: float) -> float:
    return float(np.clip(x, 0.0, 1.0))


def _scores(f: dict) -> dict[str, float]:
    """f = Twin state as dict (from features.py / engine asdict)."""
    savings = f.get("savings_rate", 0.0)
    volatility = f.get("spending_volatility", 0.0)
    income_stab = f.get("income_stability", 0.0)
    debt = f.get("debt_ratio", 0.0)
    ef_months = f.get("emergency_fund_months", 0.0)
    weekend = f.get("weekend_spending_ratio", 0.0)
    cash = f.get("cash_usage_ratio", 0.0)
    overdrafts = f.get("overdraft_months", 0)
    n_recurring = len(f.get("recurring_payments", []))

    return {
        "financially_disciplined": np.mean([
            _clip(savings / 0.25),               # saves 25%+ of income
            _clip(1 - volatility),               # predictable spending
            1.0 if overdrafts == 0 else 0.0,     # never overdrawn
            _clip(ef_months / 6),                # solid emergency fund
        ]),
        "balanced_saver": np.mean([
            _clip(savings / 0.10),               # saves, modestly
            _clip(1 - abs(savings - 0.10) / 0.10),
            _clip(income_stab),
            _clip(1 - debt / 0.30),
        ]),
        "goal_oriented_planner": np.mean([
            _clip(n_recurring / 5),              # structured commitments
            _clip(income_stab),
            _clip(1 - volatility),
            _clip(savings / 0.15),
        ]),
        "conservative_spender": np.mean([
            _clip(1 - volatility * 2),           # very steady spending
            _clip(1 - weekend * 2),              # weekday-routine spender
            _clip(1 - debt / 0.15),              # avoids credit
            _clip(cash / 0.5),                   # prefers cash
        ]),
        "impulse_shopper": np.mean([
            _clip(weekend / 0.45),               # weekend-heavy spending
            _clip(volatility / 1.0),             # erratic amounts
            _clip(1 - savings / 0.05),           # saves little
        ]),
        "risk_tolerant": np.mean([
            _clip(debt / 0.35),                  # comfortable with leverage
            _clip(savings / 0.05 if savings > 0 else 0),  # but still solvent
            _clip(1 - ef_months / 3),            # thin buffer by choice
        ]),
        "at_risk": np.mean([
            _clip(overdrafts / 3),
            _clip(-savings / 0.10),              # spending exceeds income
            _clip(debt / 0.45),
            _clip(1 - ef_months / 2),
        ]),
    }


def classify(features: dict) -> dict:
    """Return {personality, confidence, scores} — fully explainable."""
    scores = {k: round(float(v), 4) for k, v in _scores(features).items()}
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_name, top = ranked[0]
    second = ranked[1][1]
    confidence = round(top / (top + second), 3) if (top + second) else 0.5
    return {
        "personality": top_name,
        "confidence": confidence,
        "scores": scores,
    }


def risk_level(f: dict) -> str:
    """Low / medium / high risk from Twin state (used in snapshots)."""
    months_to_zero = (f.get("forecast") or {}).get("months_to_zero")
    if (f.get("financial_health_score", 100) < 35
            or f.get("debt_ratio", 0) > 0.40
            or (months_to_zero is not None and months_to_zero <= 12)
            or f.get("savings_rate", 0) < -0.05):
        return "high"
    if (f.get("financial_health_score", 0) >= 60
            and f.get("emergency_fund_months", 0) >= 6
            and f.get("debt_ratio", 1) <= 0.20):
        return "low"
    return "medium"


### `twin/diff.py` — semantic state-difference engine (powers explainability)

In [ ]:
%%writefile twin/diff.py
"""TwinDiff — semantic state-difference engine.

A diff is not just numbers: each changed attribute carries a REASON derived
from the Twin's dependency graph (which upstream attributes moved). This
single object powers explainability, the UI before/after view, and the
judge inspection output.
"""

# attribute → the upstream attributes it is derived from
DEPENDENCY_GRAPH = {
    "financial_health_score": ["savings_rate", "cashflow_stability",
                               "debt_ratio", "emergency_fund_months"],
    "savings_rate": ["monthly_income", "monthly_expenses"],
    "net_cashflow": ["monthly_income", "monthly_expenses"],
    "debt_ratio": ["monthly_loan_payment", "monthly_income"],
    "emergency_fund_months": ["current_balance", "monthly_expenses"],
    "risk_level": ["financial_health_score", "debt_ratio", "savings_rate"],
    "financial_personality": ["savings_rate", "debt_ratio",
                              "emergency_fund_months"],
    "personality_confidence": ["financial_personality"],
}

# upstream attribute + direction → human reason
REASON_PHRASES = {
    ("monthly_loan_payment", +1): "monthly obligations increased",
    ("monthly_loan_payment", -1): "monthly obligations decreased",
    ("monthly_expenses", +1): "recurring spending increased",
    ("monthly_expenses", -1): "recurring spending decreased",
    ("monthly_income", +1): "income increased",
    ("monthly_income", -1): "income decreased",
    ("current_balance", +1): "cash reserve grew",
    ("current_balance", -1): "cash reserve consumed",
    ("savings_rate", +1): "savings rate improved",
    ("savings_rate", -1): "savings rate fell",
    ("debt_ratio", +1): "debt ratio increased",
    ("debt_ratio", -1): "debt ratio decreased",
    ("emergency_fund_months", +1): "liquidity buffer grew",
    ("emergency_fund_months", -1): "liquidity buffer shrank",
    ("financial_health_score", +1): "overall health improved",
    ("financial_health_score", -1): "overall health deteriorated",
    ("cashflow_stability", +1): "cash flow became steadier",
    ("cashflow_stability", -1): "cash flow became less stable",
}


def _direction(before, after) -> int:
    if isinstance(before, (int, float)) and isinstance(after, (int, float)):
        return +1 if after > before else -1
    return 0


def _reasons_for(attr: str, before_state: dict, after_state: dict) -> list[str]:
    """Which upstream dependencies of `attr` changed, phrased as causes."""
    reasons = []
    for dep in DEPENDENCY_GRAPH.get(attr, []):
        b, a = before_state.get(dep), after_state.get(dep)
        if b is None or a is None or b == a:
            continue
        phrase = REASON_PHRASES.get((dep, _direction(b, a)))
        if phrase:
            reasons.append(phrase)
        else:
            reasons.append(f"{dep.replace('_', ' ')} changed")
    return reasons


def twin_diff(before_state: dict, after_state: dict) -> list[dict]:
    """Compute the semantic difference between two Twin states.

    Returns a list of change records, root causes first (attributes that
    are pure inputs come before derived ones, so the causal story reads
    top-down).
    """
    changes = []
    for attr, before in before_state.items():
        after = after_state.get(attr)
        if after == before:
            continue
        record = {
            "attribute": attr,
            "before": before,
            "after": after,
            "delta": round(after - before, 4)
            if isinstance(before, (int, float)) and isinstance(after, (int, float))
            else None,
            "reasons": _reasons_for(attr, before_state, after_state),
        }
        changes.append(record)

    # root causes (no dependencies among snapshot keys) first, derived last
    def depth(attr: str, seen=None) -> int:
        seen = seen or set()
        if attr in seen or attr not in DEPENDENCY_GRAPH:
            return 0
        return 1 + max((depth(d, seen | {attr})
                        for d in DEPENDENCY_GRAPH[attr]), default=0)

    changes.sort(key=lambda c: depth(c["attribute"]))
    return changes


def render(diff: list[dict], fmt) -> list[str]:
    """Human lines: 'Health Score 82 → 74 — because debt ratio increased'."""
    lines = []
    for c in diff:
        line = (f"{c['attribute'].replace('_', ' ')}: "
                f"{fmt(c['attribute'], c['before'])} → "
                f"{fmt(c['attribute'], c['after'])}")
        if c["reasons"]:
            line += "  — because " + " and ".join(c["reasons"])
        lines.append(line)
    return lines


### `twin/explain.py` — turns a diff into a human WHY narrative, mechanically (no LLM)

In [ ]:
%%writefile twin/explain.py
"""Explainability layer.

Turns a simulation's attribute-level diff into a human explanation that
references WHICH Twin attributes changed and WHY. No LLM required — the
explanation is derived mechanically from the Twin state diff, so it is
always faithful to the numbers. (An LLM layer can rephrase this text,
but never invents the reasoning.)
"""

LABELS = {
    "monthly_income": "monthly income",
    "monthly_expenses": "monthly expenses",
    "net_cashflow": "net cash flow",
    "savings_rate": "savings rate",
    "debt_ratio": "debt-to-income ratio",
    "current_balance": "account balance",
    "emergency_fund_months": "emergency fund",
    "financial_health_score": "Health Score",
    "monthly_loan_payment": "monthly loan obligations",
    "financial_personality": "financial personality",
    "risk_level": "risk level",
}

PCT_ATTRS = {"savings_rate", "debt_ratio"}
MONEY_ATTRS = {"monthly_income", "monthly_expenses", "net_cashflow",
               "current_balance", "monthly_loan_payment"}


def _fmt(attr: str, v) -> str:
    if attr in PCT_ATTRS:
        return f"{v * 100:.1f}%"
    if attr in MONEY_ATTRS:
        return f"{v:,.0f}"
    if attr == "emergency_fund_months":
        return f"{v:.1f} months"
    return str(v)


def _cause_sentence(event: dict) -> str:
    t = event.get("type")
    if t == "new_loan":
        return (f"the loan adds {event.get('monthly_payment', 0):,.0f}/month "
                f"in obligations for {event.get('duration_months')} months")
    if t == "one_off_expense":
        return f"a one-time payment of {event.get('amount', 0):,.0f} left the account"
    if t == "new_subscription":
        return (f"a new recurring commitment of "
                f"{event.get('monthly_amount', 0):,.0f}/month was added")
    if t == "salary_change":
        return "monthly income changed"
    if t == "rent_change":
        d = event.get("delta", 0)
        return f"housing costs {'rose' if d > 0 else 'fell'} by {abs(d):,.0f}/month"
    if t == "investment":
        return (f"{event.get('monthly_amount', 0):,.0f}/month now flows "
                f"into investments instead of the account")
    if t == "loan_payoff":
        return f"monthly obligations dropped by {event.get('monthly_payment', 0):,.0f}"
    return "the simulated decision was applied"


def explain(result: dict) -> str:
    """Build the WHY narrative for one simulation result.

    Single source of truth: consumes the TwinDiff already computed for the
    result (final state vs original state) — never recomputes differences.
    """
    changes = {c["attribute"]: c for c in result.get("twin_diff", [])}
    causes = "; ".join(_cause_sentence(e) for e in result.get("events_applied", []))

    effects = []
    for attr in ("debt_ratio", "savings_rate", "emergency_fund_months",
                 "current_balance", "net_cashflow"):
        if attr in changes:
            c = changes[attr]
            direction = "rose" if (c["delta"] or 0) > 0 else "fell"
            effects.append(f"your {LABELS[attr]} {direction} from "
                           f"{_fmt(attr, c['before'])} to {_fmt(attr, c['after'])}")

    health = changes.get("financial_health_score")
    lead = ""
    if health:
        verb = "decreased" if health["delta"] < 0 else "improved"
        lead = (f"The Health Score {verb} from {health['before']} to "
                f"{health['after']} because {causes}")
    else:
        lead = f"The decision was applied: {causes}"

    sentence = lead
    if effects:
        sentence += " — as a result, " + ", ".join(effects)
    risk = changes.get("risk_level")
    if risk:
        sentence += f". Risk level moved from {risk['before']} to {risk['after']}"
    return sentence + "."


def before_after_card(result: dict) -> dict:
    """Frontend-ready before/after comparison of the headline attributes."""
    b, a = result["before"], result["after"]
    card = {}
    for attr in ("financial_health_score", "savings_rate",
                 "emergency_fund_months", "debt_ratio", "risk_level",
                 "financial_personality"):
        if attr in b:
            card[attr] = {"before": _fmt(attr, b[attr]),
                          "after": _fmt(attr, a[attr]),
                          "changed": b[attr] != a[attr]}
    return card


### `twin/validation.py` — consistency invariants checked before any result is returned

In [ ]:
%%writefile twin/validation.py
"""Twin consistency checks.

Run before any simulation result is returned. Each check enforces a
financial/mathematical invariant; violations come back as structured
warnings (never silent, never fatal — judges can see the Twin policing
itself).
"""


def validate(state: dict) -> list[dict]:
    warnings = []

    def warn(check: str, message: str, severity: str = "warning"):
        warnings.append({"check": check, "severity": severity, "message": message})

    income = state.get("monthly_income", 0)
    expenses = state.get("monthly_expenses", 0)
    net = state.get("net_cashflow", 0)
    savings = state.get("savings_rate", 0)
    debt = state.get("debt_ratio", 0)
    ef = state.get("emergency_fund_months", 0)
    balance = state.get("current_balance", 0)

    # savings cannot exceed income
    if savings > 1.0:
        warn("savings_bound", f"savings rate {savings:.2f} exceeds 100% of income",
             "error")

    # cash flow must balance: net == income - expenses
    if abs(net - (income - expenses)) > 0.01:
        warn("cashflow_balance",
             f"net cashflow {net:,.2f} != income - expenses "
             f"({income - expenses:,.2f})", "error")

    # debt ratio mathematically valid
    if debt < 0:
        warn("debt_ratio_range", f"debt ratio {debt:.2f} is negative", "error")
    elif debt > 1.0:
        warn("debt_ratio_range",
             f"debt ratio {debt:.2f} exceeds income — obligations are "
             "larger than earnings")

    # emergency fund cannot be negative without explanation
    if ef < 0:
        if balance < 0:
            warn("emergency_fund_negative",
                 f"emergency fund is negative ({ef:.1f} months) because the "
                 f"account balance is overdrawn ({balance:,.0f})", "info")
        else:
            warn("emergency_fund_negative",
                 f"emergency fund {ef:.1f} months is negative with a "
                 "non-negative balance — inconsistent state", "error")

    # income sanity
    if income < 0:
        warn("income_range", f"monthly income {income:,.2f} is negative", "error")

    return warnings


### `twin/report.py` — the full structured health report

In [ ]:
%%writefile twin/report.py
"""Twin Health Report — the full structured readout of one Twin state.

Generated after every simulation (and on demand for the live Twin).
Sections follow the hackathon spec: health, cash flow, risk, liquidity,
debt, savings, personality, stability, goals, confidence, summary.
"""
from .validation import validate


def _grade(score: float) -> str:
    if score >= 75: return "excellent"
    if score >= 55: return "good"
    if score >= 35: return "fair"
    return "poor"


def health_report(state: dict) -> dict:
    """state = twin.to_dict() or a simulation's after-state merged dict."""
    score = state.get("financial_health_score", 0)
    income = state.get("monthly_income", 0)
    forecast = state.get("forecast") or {}
    months_hist = state.get("months_of_history", 0)
    goals = state.get("goals") or []

    report = {
        "financial_health": {
            "score": score,
            "grade": _grade(score),
        },
        "cash_flow": {
            "monthly_income": income,
            "monthly_expenses": state.get("monthly_expenses", 0),
            "net": state.get("net_cashflow", 0),
            "direction": "positive" if state.get("net_cashflow", 0) >= 0
                         else "negative",
        },
        "risk": {
            "level": state.get("risk_level", "medium"),
            "months_to_zero_balance": forecast.get("months_to_zero"),
            "overdraft_months_in_history": state.get("overdraft_months", 0),
        },
        "liquidity": {
            "current_balance": state.get("current_balance", 0),
            "emergency_fund_months": state.get("emergency_fund_months", 0),
            "target_months": 6,
            "status": "funded" if state.get("emergency_fund_months", 0) >= 6
                      else "underfunded",
        },
        "debt": {
            "monthly_obligations": state.get("monthly_loan_payment", 0),
            "debt_to_income": state.get("debt_ratio", 0),
            "status": "healthy" if state.get("debt_ratio", 0) <= 0.30
                      else "stretched",
        },
        "savings": {
            "rate": state.get("savings_rate", 0),
            "benchmark": 0.20,
            "status": "on_track" if state.get("savings_rate", 0) >= 0.10
                      else "behind",
        },
        "financial_personality": {
            "type": state.get("financial_personality"),
            "confidence": state.get("personality_confidence"),
        },
        "financial_stability": {
            "income_stability": state.get("income_stability", 0),
            "cashflow_stability": state.get("cashflow_stability", 0),
            "spending_volatility": state.get("spending_volatility", 0),
        },
        "goal_progress": {
            "goals": goals,
            "status": "no_goals_set" if not goals else "tracking",
        },
        "confidence": {
            # how much history backs this Twin — more months, more trust
            "months_of_history": months_hist,
            "data_confidence": round(min(1.0, months_hist / 24), 2),
            "personality_confidence": state.get("personality_confidence"),
        },
        "validation": validate(state),
    }
    report["summary"] = _summary(report)
    return report


def _summary(r: dict) -> str:
    fh, cf, li, de, sa = (r["financial_health"], r["cash_flow"],
                          r["liquidity"], r["debt"], r["savings"])
    parts = [
        f"Financial health is {fh['grade']} ({fh['score']}/100)",
        f"cash flow is {cf['direction']} at {cf['net']:,.0f}/month",
        f"emergency fund covers {li['emergency_fund_months']:.1f} of the "
        f"recommended {li['target_months']} months",
        f"debt takes {de['debt_to_income'] * 100:.0f}% of income",
        f"savings rate is {sa['rate'] * 100:.0f}%",
    ]
    issues = [w for w in r["validation"] if w["severity"] == "error"]
    if issues:
        parts.append(f"{len(issues)} consistency issue(s) flagged")
    return "; ".join(parts) + "."


### `twin/features.py` — feature engineering: raw cleaned transactions → ~20 documented behavioral features

In [ ]:
%%writefile twin/features.py
"""Feature engineering layer.

Turns cleaned transactions into per-account behavioral features.
Every feature here feeds a FinancialTwin attribute — nothing downstream
touches raw transactions.

Feature documentation
---------------------
monthly_income          median of monthly credit inflow (salary + pension + transfers in)
monthly_expenses        median of monthly debit outflow
net_cashflow            income - expenses (median month)
savings_rate            net_cashflow / income — core solvency signal
spending_volatility     std/mean of monthly expenses — lifestyle predictability
cashflow_stability      1 - CV of monthly net cashflow (clipped 0..1)
income_stability        1 - CV of monthly income — salaried vs irregular earner
debt_ratio              monthly loan payments / monthly income (DTI)
emergency_fund_months   current balance / monthly expenses — months of runway
recurring_payments      standing orders + detected recurring debits
weekend_spending_ratio  share of debit volume on Fri/Sat/Sun — impulse proxy
cash_usage_ratio        share of spending done in cash withdrawals
overdraft_events        count of negative-balance months — distress signal
financial_health_score  weighted composite (0-100), weights in config
"""
import numpy as np
import pandas as pd

from . import config


def _cv(series: pd.Series) -> float:
    """Coefficient of variation, safe against zero mean."""
    m = series.mean()
    return float(series.std() / m) if m else 0.0


def detect_salary(acct_tx: pd.DataFrame) -> float:
    """Detect recurring monthly income: large credits arriving ~monthly."""
    credits = acct_tx[(acct_tx["trans_type"] == "credit")
                      & (acct_tx["amount"] >= config.SALARY_MIN_AMOUNT)]
    if credits.empty:
        return 0.0
    # group similar amounts (within 5%) and check monthly recurrence
    by_month = credits.groupby("month")["amount"].max()
    if len(by_month) < config.RECURRING_MIN_OCCURRENCES:
        return 0.0
    return float(by_month.median())


def detect_recurring_debits(acct_tx: pd.DataFrame) -> list[dict]:
    """Find debits with same rounded amount recurring >= N months (subscriptions, bills)."""
    debits = acct_tx[acct_tx["trans_type"].isin(["debit", "cash_withdrawal"])].copy()
    if debits.empty:
        return []
    debits["amount_bucket"] = debits["amount"].round(0)
    out = []
    for (bucket, cat), grp in debits.groupby(["amount_bucket", "category"]):
        months = grp["month"].nunique()
        if months >= config.RECURRING_MIN_OCCURRENCES and bucket > 0:
            out.append({
                "amount": float(bucket),
                "category": cat,
                "months_observed": int(months),
                "monthly": bool(months >= grp["month"].nunique() * 0.8),
            })
    # keep the strongest recurring signals
    out.sort(key=lambda d: d["months_observed"], reverse=True)
    return out[:10]


def monthly_aggregates(acct_tx: pd.DataFrame) -> pd.DataFrame:
    """Per-month income / expenses / net / end balance for one account."""
    g = acct_tx.groupby("month")
    agg = pd.DataFrame({
        "income": g.apply(lambda x: x.loc[x["signed_amount"] > 0, "signed_amount"].sum(),
                          include_groups=False),
        "expenses": g.apply(lambda x: -x.loc[x["signed_amount"] < 0, "signed_amount"].sum(),
                            include_groups=False),
        "end_balance": g.apply(lambda x: x.sort_values("trans_date")["balance"].iloc[-1],
                               include_groups=False),
    })
    agg["net"] = agg["income"] - agg["expenses"]
    return agg


def category_ratios(acct_tx: pd.DataFrame) -> dict:
    """Share of spending volume per normalized category."""
    debits = acct_tx[acct_tx["signed_amount"] < 0]
    total = -debits["signed_amount"].sum()
    if not total:
        return {}
    shares = (-debits.groupby("category")["signed_amount"].sum() / total)
    return {f"{cat}_ratio": round(float(v), 4) for cat, v in shares.items()}


def health_score(features: dict) -> float:
    """Composite 0-100 financial health score (documented weights in config)."""
    w = config.HEALTH_SCORE_WEIGHTS
    savings_component = np.clip(features["savings_rate"] / 0.30, 0, 1)          # 30% savings = perfect
    stability_component = np.clip(features["cashflow_stability"], 0, 1)
    debt_component = 1 - np.clip(features["debt_ratio"] / 0.40, 0, 1)           # 40% DTI = zero score
    ef_component = np.clip(
        features["emergency_fund_months"] / config.EMERGENCY_FUND_MONTHS_TARGET, 0, 1)
    score = 100 * (w["savings_rate"] * savings_component
                   + w["cashflow_stability"] * stability_component
                   + w["debt_ratio"] * debt_component
                   + w["emergency_fund"] * ef_component)
    return round(float(score), 1)


def build_account_features(acct_tx: pd.DataFrame, loans: pd.DataFrame) -> dict:
    """All behavioral features for a single account."""
    monthly = monthly_aggregates(acct_tx)
    if monthly.empty:
        return {}

    income_m = float(monthly["income"].median())
    expenses_m = float(monthly["expenses"].median())
    current_balance = float(
        acct_tx.sort_values("trans_date")["balance"].iloc[-1])

    active_loans = loans[loans["status"].isin(["running_ok", "running_in_debt"])]
    monthly_loan_payment = float(active_loans["payments"].sum())

    features = {
        "monthly_income": round(income_m, 2),
        "monthly_expenses": round(expenses_m, 2),
        "net_cashflow": round(income_m - expenses_m, 2),
        "savings_rate": round((income_m - expenses_m) / income_m, 4) if income_m else 0.0,
        "spending_volatility": round(_cv(monthly["expenses"]), 4),
        "income_stability": round(float(np.clip(1 - _cv(monthly["income"]), 0, 1)), 4),
        "cashflow_stability": round(float(np.clip(1 - _cv(monthly["net"].abs() + 1), 0, 1)), 4),
        "debt_ratio": round(monthly_loan_payment / income_m, 4) if income_m else 0.0,
        "monthly_loan_payment": round(monthly_loan_payment, 2),
        "current_balance": round(current_balance, 2),
        "emergency_fund_months": round(current_balance / expenses_m, 2) if expenses_m else 0.0,
        "detected_salary": round(detect_salary(acct_tx), 2),
        "weekend_spending_ratio": _weekend_ratio(acct_tx),
        "cash_usage_ratio": _cash_ratio(acct_tx),
        "overdraft_months": int((monthly["end_balance"] < 0).sum()),
        "months_of_history": int(len(monthly)),
    }
    features.update(category_ratios(acct_tx))
    features["financial_health_score"] = health_score(features)
    features["recurring_payments"] = detect_recurring_debits(acct_tx)
    return features


def _weekend_ratio(acct_tx: pd.DataFrame) -> float:
    debits = acct_tx[acct_tx["signed_amount"] < 0]
    total = -debits["signed_amount"].sum()
    if not total:
        return 0.0
    weekend = -debits[debits["trans_date"].dt.dayofweek >= 4]["signed_amount"].sum()
    return round(float(weekend / total), 4)


def _cash_ratio(acct_tx: pd.DataFrame) -> float:
    debits = acct_tx[acct_tx["signed_amount"] < 0]
    total = -debits["signed_amount"].sum()
    if not total:
        return 0.0
    cash = -debits[debits["operation"] == "cash_withdrawal"]["signed_amount"].sum()
    return round(float(cash / total), 4)


### `twin/memory.py` — milestone timeline extracted from account history

In [ ]:
%%writefile twin/memory.py
"""Twin Memory — milestone timeline extracted from account history.

A Digital Twin remembers what happened to it. This module scans the
cleaned history once at build time and produces a chronological list of
significant financial events; afterwards `engine.apply_event()` appends
live events to the same timeline.

Milestone types detected from history:
  salary_detected     first month a recurring salary appears
  salary_change       sustained shift (>15%) in recurring income
  loan_granted        from the loans table (real outcome attached)
  loan_finished       loan contract ended (repaid or defaulted)
  large_purchase      debit > LARGE_PURCHASE_FACTOR x median debit
  first_overdraft     first month balance went negative
"""
import pandas as pd

LARGE_PURCHASE_FACTOR = 8
SALARY_SHIFT_THRESHOLD = 0.15
MAX_LARGE_PURCHASES = 5


def _month_str(period) -> str:
    return str(period)


def build_timeline(acct_tx: pd.DataFrame, acct_loans: pd.DataFrame) -> list[dict]:
    events: list[dict] = []

    # --- salary detection & changes ---
    credits = acct_tx[acct_tx["signed_amount"] > 0]
    if not credits.empty:
        monthly_max = credits.groupby("month")["signed_amount"].max()
        if len(monthly_max) >= 3:
            first = monthly_max.index[0]
            events.append({
                "date": _month_str(first), "type": "salary_detected",
                "title": "Recurring income detected",
                "amount": round(float(monthly_max.iloc[:3].median()), 2)})
            # sustained shifts: compare 3-month rolling medians
            rolled = monthly_max.rolling(3).median().dropna()
            prev = rolled.iloc[0]
            for period, val in rolled.items():
                if prev and abs(val - prev) / prev > SALARY_SHIFT_THRESHOLD:
                    events.append({
                        "date": _month_str(period),
                        "type": "salary_change",
                        "title": f"Income {'increased' if val > prev else 'decreased'}",
                        "amount": round(float(val), 2),
                        "previous": round(float(prev), 2)})
                    prev = val

    # --- loans (with real outcomes) ---
    for _, l in acct_loans.iterrows():
        events.append({
            "date": str(l["granted_date"].date()) if pd.notna(l["granted_date"]) else None,
            "type": "loan_granted",
            "title": f"Loan granted ({l['duration']:.0f} months)",
            "amount": round(float(l["amount"]), 2),
            "monthly_payment": round(float(l["payments"]), 2),
            "outcome": l["status"]})
        if l["status"] in ("finished_ok", "defaulted"):
            events.append({
                "date": None, "type": "loan_finished",
                "title": "Loan repaid in full" if l["status"] == "finished_ok"
                         else "Loan defaulted",
                "amount": round(float(l["amount"]), 2),
                "outcome": l["status"]})

    # --- large purchases ---
    debits = acct_tx[acct_tx["signed_amount"] < 0]
    if not debits.empty:
        median_debit = debits["amount"].median()
        big = debits[debits["amount"] > LARGE_PURCHASE_FACTOR * median_debit]
        big = big.nlargest(MAX_LARGE_PURCHASES, "amount")
        for _, r in big.iterrows():
            events.append({
                "date": str(r["trans_date"].date()),
                "type": "large_purchase",
                "title": f"Large {r['category']} payment",
                "amount": round(float(r["amount"]), 2)})

    # --- first overdraft ---
    negative = acct_tx[acct_tx["balance"] < 0].sort_values("trans_date")
    if not negative.empty:
        events.append({
            "date": str(negative.iloc[0]["trans_date"].date()),
            "type": "first_overdraft",
            "title": "Account went into overdraft",
            "amount": round(float(negative.iloc[0]["balance"]), 2)})

    events.sort(key=lambda e: e["date"] or "9999")
    return events


### `twin/engine.py` — the Twin itself: persistent state, `apply_event()`, forecast

In [ ]:
%%writefile twin/engine.py
"""FinancialTwin engine.

The Twin is a living state object: built once from history (via features.py),
then updated incrementally by `apply_event()` — every new transaction, loan,
salary change, or simulated decision mutates the state and recomputes the
derived attributes. The AI layer reasons from `to_dict()`, never from raw
transactions.
"""
from dataclasses import dataclass, field, asdict
from datetime import datetime

from .features import health_score
from .personality import classify, risk_level


@dataclass
class FinancialTwin:
    """Continuously-updated virtual representation of one person's finances."""
    account_id: int
    monthly_income: float = 0.0
    monthly_expenses: float = 0.0
    net_cashflow: float = 0.0
    savings_rate: float = 0.0
    spending_volatility: float = 0.0
    income_stability: float = 0.0
    cashflow_stability: float = 0.0
    debt_ratio: float = 0.0
    monthly_loan_payment: float = 0.0
    current_balance: float = 0.0
    emergency_fund_months: float = 0.0
    detected_salary: float = 0.0
    weekend_spending_ratio: float = 0.0
    cash_usage_ratio: float = 0.0
    overdraft_months: int = 0
    months_of_history: int = 0
    financial_health_score: float = 0.0
    financial_personality: str = "balanced_saver"
    personality_confidence: float = 0.5
    personality_scores: dict = field(default_factory=dict)
    risk_level: str = "medium"
    memory: list = field(default_factory=list)
    recurring_payments: list = field(default_factory=list)
    category_ratios: dict = field(default_factory=dict)
    demographics: dict = field(default_factory=dict)
    goals: list = field(default_factory=list)
    forecast: dict = field(default_factory=dict)
    last_updated: str = ""
    event_log: list = field(default_factory=list)

    # ---------- construction ----------

    @classmethod
    def from_features(cls, account_id: int, features: dict,
                      demographics: dict | None = None,
                      memory: list | None = None) -> "FinancialTwin":
        ratios = {k: v for k, v in features.items() if k.endswith("_ratio")
                  and k not in ("debt_ratio", "weekend_spending_ratio", "cash_usage_ratio")}
        core = {k: v for k, v in features.items()
                if k in cls.__dataclass_fields__ and not k.endswith("_ratio")
                or k in ("debt_ratio", "weekend_spending_ratio", "cash_usage_ratio")}
        twin = cls(account_id=account_id, **core)
        twin.category_ratios = ratios
        twin.demographics = demographics or {}
        twin.memory = memory or []
        twin._refresh()
        return twin

    # ---------- derived state ----------

    def _refresh(self):
        """Recompute every derived attribute after any state change."""
        self.net_cashflow = round(self.monthly_income - self.monthly_expenses, 2)
        self.savings_rate = round(self.net_cashflow / self.monthly_income, 4) \
            if self.monthly_income else 0.0
        self.debt_ratio = round(self.monthly_loan_payment / self.monthly_income, 4) \
            if self.monthly_income else 0.0
        self.emergency_fund_months = round(self.current_balance / self.monthly_expenses, 2) \
            if self.monthly_expenses else 0.0
        state = asdict(self)
        self.financial_health_score = health_score(state)
        state["financial_health_score"] = self.financial_health_score
        p = classify(state)
        self.financial_personality = p["personality"]
        self.personality_confidence = p["confidence"]
        self.personality_scores = p["scores"]
        self.forecast = self._forecast()
        state["forecast"] = self.forecast
        self.risk_level = risk_level(state)
        self.last_updated = datetime.now().isoformat(timespec="seconds")

    def _forecast(self, horizon_months: int = 24) -> dict:
        """Simple deterministic projection of balance under current behavior."""
        balances = []
        b = self.current_balance
        for _ in range(horizon_months):
            b += self.net_cashflow
            balances.append(round(b, 2))
        return {
            "horizon_months": horizon_months,
            "projected_balances": balances,
            "balance_in_12m": balances[11],
            "balance_in_24m": balances[23],
            "months_to_zero": next(
                (i + 1 for i, v in enumerate(balances) if v < 0), None),
        }

    # ---------- live updates ----------

    def apply_event(self, event: dict) -> dict:
        """Update the Twin from a financial event. Returns a change report.

        event = {"type": ..., **params} — types:
          transaction        {amount (+/-), category}
          salary_change      {new_salary} or {delta}
          new_loan           {principal, duration_months, annual_rate}
          loan_payoff        {monthly_payment}
          new_subscription   {monthly_amount, name}
          cancel_subscription{monthly_amount}
          rent_change        {delta}
          one_off_expense    {amount}
          investment         {monthly_amount}
        """
        before = self.snapshot()
        etype = event["type"]

        if etype == "transaction":
            self.current_balance += event["amount"]
            # running-average update of monthly expense profile
            if event["amount"] < 0 and self.months_of_history:
                self.monthly_expenses += abs(event["amount"]) / max(self.months_of_history, 1)

        elif etype == "salary_change":
            new = event.get("new_salary", self.monthly_income + event.get("delta", 0))
            self.monthly_income = round(new, 2)
            self.detected_salary = self.monthly_income

        elif etype == "new_loan":
            payment = amortized_payment(
                event["principal"], event.get("annual_rate", 0.06),
                event["duration_months"])
            self.monthly_loan_payment += payment
            # historical loan payments arrive as transactions and are already
            # inside monthly_expenses; a simulated loan must add its payment
            # here too, or cashflow and the 24m forecast would ignore it
            self.monthly_expenses += payment
            self.current_balance += event.get("disbursed_to_account", 0)
            event["monthly_payment"] = round(payment, 2)

        elif etype == "loan_payoff":
            self.monthly_loan_payment = max(
                0.0, self.monthly_loan_payment - event["monthly_payment"])
            self.monthly_expenses = max(
                0.0, self.monthly_expenses - event["monthly_payment"])

        elif etype == "new_subscription":
            self.monthly_expenses += event["monthly_amount"]
            self.recurring_payments.append({
                "amount": event["monthly_amount"],
                "category": event.get("name", "subscription"),
                "months_observed": 0, "monthly": True})

        elif etype == "cancel_subscription":
            self.monthly_expenses = max(
                0.0, self.monthly_expenses - event["monthly_amount"])

        elif etype == "rent_change":
            self.monthly_expenses += event["delta"]

        elif etype == "one_off_expense":
            self.current_balance -= event["amount"]

        elif etype == "investment":
            self.monthly_expenses += event["monthly_amount"]  # cash out of account

        else:
            raise ValueError(f"unknown event type: {etype}")

        self._refresh()
        after = self.snapshot()
        report = diff_snapshots(before, after)
        self.event_log.append({"event": event, "changes": report,
                               "at": self.last_updated})
        # the Twin remembers: live events join the historical timeline
        self.memory.append({
            "date": self.last_updated[:10], "type": etype,
            "title": etype.replace("_", " "),
            "amount": event.get("amount") or event.get("monthly_amount")
                      or event.get("principal") or event.get("delta"),
            "source": "live"})
        return report

    # ---------- output ----------

    def snapshot(self) -> dict:
        keys = ["monthly_income", "monthly_expenses", "net_cashflow", "savings_rate",
                "debt_ratio", "current_balance", "emergency_fund_months",
                "financial_health_score", "financial_personality",
                "personality_confidence", "risk_level", "monthly_loan_payment"]
        return {k: getattr(self, k) for k in keys}

    def to_dict(self) -> dict:
        d = asdict(self)
        d.pop("event_log", None)
        return d


def amortized_payment(principal: float, annual_rate: float, months: int) -> float:
    """Standard amortized monthly payment."""
    if months <= 0:
        return 0.0
    r = annual_rate / 12
    if r == 0:
        return principal / months
    return principal * r * (1 + r) ** months / ((1 + r) ** months - 1)


def diff_snapshots(before: dict, after: dict) -> list[dict]:
    """Which Twin attributes changed and by how much — feeds AI explanations."""
    changes = []
    for k, b in before.items():
        a = after[k]
        if a != b:
            changes.append({"attribute": k, "before": b, "after": a,
                            "delta": round(a - b, 4) if isinstance(a, (int, float)) else None})
    return changes


### `twin/simulation.py` — what-if engine: deep-copies the Twin, never mutates the real one

In [ ]:
%%writefile twin/simulation.py
"""What-if Simulation Engine.

Every simulation:
  1. deep-copies the Twin (the real Twin is never mutated by a what-if)
  2. applies the decision as events
  3. returns before/after states + attribute-level diff + verdict

This is the layer the AI recommendation engine calls.
"""
import copy

from dataclasses import asdict

from .diff import twin_diff
from .engine import FinancialTwin
from .explain import explain, before_after_card
from .report import health_report
from .validation import validate


# verdict thresholds on the *post-decision* twin
def _verdict(twin: FinancialTwin) -> str:
    if twin.forecast.get("months_to_zero") and twin.forecast["months_to_zero"] <= 12:
        return "dangerous"
    if twin.debt_ratio > 0.40 or twin.savings_rate < 0:
        return "risky"
    if twin.savings_rate < 0.10 or twin.emergency_fund_months < 3:
        return "caution"
    return "safe"


class SimulationEngine:
    def __init__(self, twin: FinancialTwin):
        self.twin = twin

    def _run(self, events: list[dict], label: str) -> dict:
        sim = copy.deepcopy(self.twin)
        sim.event_log = []
        all_changes = []
        for e in events:
            all_changes.extend(sim.apply_event(e))

        before, after = self.twin.snapshot(), sim.snapshot()
        result = {
            "simulation": label,
            "verdict": _verdict(sim),
            # explicit state transition: State A → event(s) → State B
            "transition": {
                "from_state": before,
                "events": events,
                "to_state": after,
            },
            "before": before,
            "after": after,
            "twin_diff": twin_diff(before, after),
            "changed_attributes": all_changes,
            "forecast_after": sim.forecast,
            "events_applied": events,
            # consistency check on the post-event Twin, never skipped
            "validation": validate(asdict(sim)),
            "health_report": health_report(asdict(sim)),
        }
        result["explanation"] = explain(result)
        result["before_after_card"] = before_after_card(result)
        return result

    # ---- supported what-if scenarios ----

    def buy_item(self, price: float, label: str = "purchase") -> dict:
        """One-off purchase paid from balance (electronics, furniture...)."""
        return self._run([{"type": "one_off_expense", "amount": price}],
                         f"buy_{label}_{price:.0f}")

    def buy_with_bnpl(self, price: float, installments: int = 4) -> dict:
        """BNPL: split into equal monthly installments, no interest."""
        monthly = price / installments
        return self._run(
            [{"type": "new_subscription", "monthly_amount": monthly,
              "name": f"bnpl_{installments}x"}],
            f"bnpl_{price:.0f}_over_{installments}m")

    def take_loan(self, principal: float, duration_months: int,
                  annual_rate: float = 0.06, disbursed: bool = True) -> dict:
        """Bank loan: monthly amortized payment + optional cash-in."""
        return self._run(
            [{"type": "new_loan", "principal": principal,
              "duration_months": duration_months, "annual_rate": annual_rate,
              "disbursed_to_account": principal if disbursed else 0}],
            f"loan_{principal:.0f}_{duration_months}m")

    def buy_car(self, price: float, down_payment_pct: float = 0.20,
                duration_months: int = 60, annual_rate: float = 0.05) -> dict:
        down = price * down_payment_pct
        financed = price - down
        return self._run(
            [{"type": "one_off_expense", "amount": down},
             {"type": "new_loan", "principal": financed,
              "duration_months": duration_months, "annual_rate": annual_rate,
              "disbursed_to_account": 0}],
            f"car_{price:.0f}")

    def new_subscription(self, monthly_amount: float, name: str = "subscription") -> dict:
        return self._run(
            [{"type": "new_subscription", "monthly_amount": monthly_amount, "name": name}],
            f"subscription_{name}")

    def rent_increase(self, delta: float) -> dict:
        return self._run([{"type": "rent_change", "delta": delta}],
                         f"rent_change_{delta:+.0f}")

    def salary_change(self, new_salary: float) -> dict:
        return self._run([{"type": "salary_change", "new_salary": new_salary}],
                         f"salary_to_{new_salary:.0f}")

    def medical_emergency(self, cost: float) -> dict:
        return self._run([{"type": "one_off_expense", "amount": cost}],
                         f"medical_{cost:.0f}")

    def vacation(self, cost: float) -> dict:
        return self._run([{"type": "one_off_expense", "amount": cost}],
                         f"vacation_{cost:.0f}")

    def invest_monthly(self, monthly_amount: float,
                       annual_return: float = 0.07, horizon_months: int = 120) -> dict:
        """Monthly investment: cash leaves the account, wealth compounds outside it."""
        result = self._run(
            [{"type": "investment", "monthly_amount": monthly_amount}],
            f"invest_{monthly_amount:.0f}_monthly")
        r = annual_return / 12
        fv = monthly_amount * (((1 + r) ** horizon_months - 1) / r)
        result["investment_projection"] = {
            "horizon_months": horizon_months,
            "total_contributed": round(monthly_amount * horizon_months, 2),
            "projected_value": round(fv, 2),
            "projected_gain": round(fv - monthly_amount * horizon_months, 2),
        }
        return result

    def payoff_debt(self, monthly_payment: float) -> dict:
        return self._run([{"type": "loan_payoff", "monthly_payment": monthly_payment}],
                         "debt_payoff")

    def compare(self, scenarios: list[dict]) -> list[dict]:
        """Run several scenarios and rank by resulting health score.

        scenarios = [{"method": "take_loan", "args": {...}}, ...]
        """
        results = []
        for s in scenarios:
            results.append(getattr(self, s["method"])(**s.get("args", {})))
        results.sort(key=lambda r: r["after"]["financial_health_score"], reverse=True)
        return results


## 2. Orchestration — `build_twins.py`, `check_architecture.py`, `api.py`

These live at the repo root, not inside `twin/`, on purpose: they're the only
code allowed to import `data_loader`/pandas directly. `check_architecture.py`
statically enforces that nothing under `twin/` ever does.

### `build_twins.py` — raw data → cleaned → features → Twin, for every account

In [ ]:
%%writefile build_twins.py
"""End-to-end pipeline: raw Berka data → cleaned → features → FinancialTwin per account.

Usage:
    python3 build_twins.py                 # build all twins, export JSON
    python3 build_twins.py --account 19    # build + demo simulations for one account
    python3 build_twins.py --inspect 19    # judge mode: full data lineage for one account
"""
import argparse
import json
import sys
import time

from twin import config
from twin.data_loader import load_all
from twin.features import build_account_features
from twin.engine import FinancialTwin
from twin.memory import build_timeline
from twin.simulation import SimulationEngine


def build_all_twins(data: dict, limit: int | None = None) -> dict[int, FinancialTwin]:
    tx = data["transactions"]
    loans = data["loans"]
    accounts = data["accounts"].merge(
        data["districts"], on="district_id", how="left")
    disp = data["dispositions"]
    clients = data["clients"]

    # owner age/gender per account
    owners = (disp[disp["disp_type"] == "O"]
              .merge(clients, on="client_id"))

    twins: dict[int, FinancialTwin] = {}
    account_ids = tx["account_id"].unique()
    if limit:
        account_ids = account_ids[:limit]

    grouped = tx.groupby("account_id")
    t0 = time.time()
    for i, acct_id in enumerate(account_ids):
        acct_tx = grouped.get_group(acct_id)
        acct_loans = loans[loans["account_id"] == acct_id]
        feats = build_account_features(acct_tx, acct_loans)
        if not feats:
            continue
        timeline = build_timeline(acct_tx, acct_loans)
        row = accounts[accounts["account_id"] == acct_id]
        owner = owners[owners["account_id"] == acct_id]
        demo = {}
        if not row.empty:
            demo = {"district": row.iloc[0]["district_name"],
                    "region": row.iloc[0]["region"],
                    "district_avg_salary": float(row.iloc[0]["avg_salary"]),
                    "district_unemployment": float(row.iloc[0]["unemp_96"])}
        if not owner.empty:
            demo["gender"] = owner.iloc[0]["gender"]
            demo["birth_year"] = int(owner.iloc[0]["birth_date"].year)
        twins[int(acct_id)] = FinancialTwin.from_features(
            int(acct_id), feats, demo, memory=timeline)
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(account_ids)} twins built "
                  f"({time.time()-t0:.0f}s)", file=sys.stderr)
    return twins


def export(twins: dict[int, FinancialTwin]):
    config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out = config.OUTPUT_DIR / "twins.json"
    with open(out, "w") as f:
        json.dump({str(k): v.to_dict() for k, v in twins.items()}, f, indent=1)
    print(f"exported {len(twins)} twins → {out}")


def demo_simulations(twin: FinancialTwin):
    sim = SimulationEngine(twin)
    print("\n=== TWIN STATE ===")
    print(json.dumps(twin.snapshot(), indent=2))
    print(f"personality: {twin.financial_personality} "
          f"(confidence {twin.personality_confidence})")
    print("\n=== TWIN MEMORY (timeline) ===")
    for e in twin.memory[:8]:
        print(f"  {e.get('date')}: {e['title']}"
              + (f" — {e['amount']:,.0f}" if e.get("amount") else ""))

    print("\n=== WHAT-IF: buy a car (20k, 20% down, 60m @5%) ===")
    r = sim.buy_car(20000)
    print(f"verdict: {r['verdict']}")
    print(f"explanation: {r['explanation']}")
    print(json.dumps(r["before_after_card"], indent=2))

    print("\n=== WHAT-IF: compare car loan vs cash purchase vs waiting ===")
    ranked = sim.compare([
        {"method": "buy_car", "args": {"price": 20000}},
        {"method": "buy_item", "args": {"price": 20000, "label": "car_cash"}},
        {"method": "invest_monthly", "args": {"monthly_amount": 350}},
    ])
    for r in ranked:
        print(f"  {r['simulation']:30s} verdict={r['verdict']:10s} "
              f"health {r['before']['financial_health_score']} → "
              f"{r['after']['financial_health_score']}")


def inspect_account(data: dict, twin: FinancialTwin, acct_id: int):
    """Judge inspection mode: raw → features → state → event → state' → why."""
    from twin.features import build_account_features
    tx = data["transactions"]
    acct_tx = tx[tx["account_id"] == acct_id]

    print("=" * 72)
    print(f"INSPECTION — account {acct_id} (full data lineage)")
    print("=" * 72)

    print("\n[1] RAW DATA (cleaned transactions feeding this Twin)")
    print(f"    {len(acct_tx):,} transactions, "
          f"{acct_tx['trans_date'].min().date()} → {acct_tx['trans_date'].max().date()}")
    print(acct_tx[["trans_date", "signed_amount", "balance", "trans_type",
                   "operation", "category"]].head(5).to_string(index=False))

    print("\n[2] ENGINEERED FEATURES (features.py — documented formulas)")
    feats = build_account_features(
        acct_tx, data["loans"][data["loans"]["account_id"] == acct_id])
    for k, v in feats.items():
        if not isinstance(v, (list, dict)):
            print(f"    {k:28s} {v}")

    print("\n[3] TWIN STATE (engine.py — derived, persistent)")
    print(json.dumps(twin.snapshot(), indent=4))
    print(f"    memory events: {len(twin.memory)}")

    print("\n[4] SIMULATION EVENT → STATE TRANSITION (simulation.py)")
    sim = SimulationEngine(twin)
    r = sim.take_loan(principal=15000, duration_months=48)
    print(f"    event: loan 15,000 over 48 months")
    print(f"    verdict: {r['verdict']}")

    print("\n[5] TWIN STATE AFTER (with semantic diff — diff.py)")
    for c in r["twin_diff"]:
        reasons = " — because " + " and ".join(c["reasons"]) if c["reasons"] else ""
        print(f"    {c['attribute']:28s} {c['before']} → {c['after']}{reasons}")

    print("\n[6] EXPLANATION (explain.py — derived from the diff, not generated)")
    print(f"    {r['explanation']}")

    print("\n[7] CONSISTENCY CHECK (validation.py)")
    if r["validation"]:
        for w in r["validation"]:
            print(f"    [{w['severity']}] {w['check']}: {w['message']}")
    else:
        print("    all invariants hold")

    print("\n[8] HEALTH REPORT SUMMARY (report.py)")
    print(f"    {r['health_report']['summary']}")

    print("\n[9] ARCHITECTURE ISOLATION PROOF (check_architecture.py)")
    from check_architecture import run_check
    run_check(verbose=True)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--account", type=int, help="run demo simulations for one account")
    ap.add_argument("--inspect", type=int, help="judge mode: full lineage for one account")
    ap.add_argument("--limit", type=int, help="build only first N accounts (dev mode)")
    args = ap.parse_args()

    print("loading + cleaning data...", file=sys.stderr)
    data = load_all()
    print(f"  transactions: {len(data['transactions']):,} rows after cleaning",
          file=sys.stderr)

    print("building twins...", file=sys.stderr)
    twins = build_all_twins(data, limit=args.limit)
    export(twins)

    if args.account and args.account in twins:
        demo_simulations(twins[args.account])
    elif args.account:
        print(f"account {args.account} not found")

    if args.inspect and args.inspect in twins:
        inspect_account(data, twins[args.inspect], args.inspect)
    elif args.inspect:
        print(f"account {args.inspect} not found")


### `check_architecture.py` — AST-based proof the simulation layer never touches raw data

In [ ]:
%%writefile check_architecture.py
"""Architecture validation — proves the simulation layer never touches raw data.

Statically inspects every module downstream of the Twin (engine, simulation,
diff, explain, personality, report, validation, memory-at-runtime) and fails
if any of them imports the data source or reads files.

Run directly, or via `build_twins.py --inspect N` (judge mode).
"""
import ast
import sys
from pathlib import Path

TWIN_DIR = Path(__file__).parent / "twin"

# modules that must be pure state-machines (no raw-data access)
DOWNSTREAM = ["engine.py", "simulation.py", "diff.py", "explain.py",
              "personality.py", "report.py", "validation.py"]

# the only module allowed to know about the data source
FORBIDDEN_IMPORTS = {"data_loader", "pandas", "csv", "sqlite3"}
FORBIDDEN_CALLS = {"read_csv", "read_json", "open"}


def check_module(path: Path) -> list[str]:
    tree = ast.parse(path.read_text())
    violations = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names]
            module = getattr(node, "module", "") or ""
            for banned in FORBIDDEN_IMPORTS:
                if banned in module or any(banned in n for n in names):
                    violations.append(
                        f"{path.name}:{node.lineno} imports '{banned}'")
        if isinstance(node, ast.Call):
            fn = node.func
            name = getattr(fn, "attr", None) or getattr(fn, "id", None)
            if name in FORBIDDEN_CALLS:
                violations.append(
                    f"{path.name}:{node.lineno} calls '{name}()'")
    return violations


def run_check(verbose: bool = True) -> bool:
    all_violations = []
    for fname in DOWNSTREAM:
        path = TWIN_DIR / fname
        if not path.exists():
            continue
        v = check_module(path)
        all_violations.extend(v)
        if verbose:
            status = "FAIL" if v else "OK  "
            print(f"  [{status}] twin/{fname} — "
                  + ("; ".join(v) if v else "reads Twin state only"))
    ok = not all_violations
    if verbose:
        print("\n  ARCHITECTURE " + ("VALID: simulation layer is fully "
              "isolated from raw transactions" if ok else "VIOLATED"))
    return ok


if __name__ == "__main__":
    sys.exit(0 if run_check() else 1)


### `verify_architecture.py` — an earlier, narrower version of the same isolation check

Kept for repo completeness. `check_architecture.py` above supersedes it (also
catches `read_csv`/`read_json`/`open()` calls, not just imports), so it isn't
re-run separately below.

In [ ]:
%%writefile verify_architecture.py
"""Architecture validation — proves the simulation layer never touches raw data.

Statically scans every module downstream of the Twin (engine, simulation,
diff, report, explain, validation, personality) and fails if any of them
imports the data layer or pandas IO. Run it live in front of judges:

    python3 verify_architecture.py
"""
import ast
import sys
from pathlib import Path

TWIN_DIR = Path(__file__).parent / "twin"

# modules that must be isolated from raw transactions
DOWNSTREAM = ["engine.py", "simulation.py", "diff.py", "report.py",
              "explain.py", "validation.py", "personality.py"]
# forbidden dependencies for those modules
FORBIDDEN = {"data_loader", "pandas", "pd"}


def imports_of(path: Path) -> set[str]:
    tree = ast.parse(path.read_text())
    found = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            found.update(a.name.split(".")[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                found.add(node.module.split(".")[-1])
            found.update(a.name for a in node.names)
    return found


def main() -> int:
    failures = []
    for name in DOWNSTREAM:
        mods = imports_of(TWIN_DIR / name)
        bad = mods & FORBIDDEN
        status = "FAIL" if bad else "OK"
        print(f"  twin/{name:18s} {status}"
              + (f"  ← imports {sorted(bad)}" if bad else ""))
        if bad:
            failures.append(name)

    print()
    if failures:
        print("ARCHITECTURE VIOLATION: simulation layer depends on raw data.")
        return 1
    print("PASS — the Twin and Simulation layers are fully isolated from raw "
          "transactions.\nThey can only reason from FinancialTwin state.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


### `api.py` — FastAPI service the Wahla frontend (and this notebook) talks to

In [ ]:
%%writefile api.py
"""HTTP API bridging the Wahla frontend to the Financial Digital Twin.

Endpoints (all JSON):
  GET  /api/accounts                     demo accounts for the connect screen
  POST /api/connect                      live Open Banking feed → cleaned → Twin
  GET  /api/twin/{account_id}            full Twin state
  POST /api/simulate/{account_id}        Wahla decision → simulation result
  GET  /api/alternatives/{account_id}    smarter-alternative suggestions

Run: python3 -m uvicorn api:app --port 8000 --reload
"""
import json

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from twin import config
from twin.data_loader import load_live_loans, load_live_transactions
from twin.engine import FinancialTwin
from twin.features import build_account_features
from twin.memory import build_timeline
from twin.simulation import SimulationEngine

app = FastAPI(title="Financial Digital Twin API")
app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"],
    allow_headers=["*"],
)

# demo personas surfaced on the connect screen (real Berka accounts)
DEMO_ACCOUNTS = [
    {"id": 21, "title": "الحساب الرئيسي", "bank": "مصرف الإنماء",
     "mask": "•••• 4821", "persona": "منضبط ماليًا"},
    {"id": 19, "title": "حساب إضافي", "bank": "مصرف الإنماء",
     "mask": "•••• 7914", "persona": "إنفاق اندفاعي"},
    {"id": 2, "title": "حساب المخطط", "bank": "مصرف الإنماء",
     "mask": "•••• 3058", "persona": "مخطط لأهدافه"},
]

_TWINS: dict[int, FinancialTwin] = {}


def _load_twins():
    path = config.OUTPUT_DIR / "twins.json"
    raw = json.loads(path.read_text())
    fields = set(FinancialTwin.__dataclass_fields__)
    for k, state in raw.items():
        state = {f: v for f, v in state.items() if f in fields}
        _TWINS[int(k)] = FinancialTwin(**state)
    print(f"loaded {len(_TWINS)} twins")


_load_twins()


def _twin(account_id: int) -> FinancialTwin:
    twin = _TWINS.get(account_id)
    if twin is None:
        raise HTTPException(404, f"no twin for account {account_id}")
    return twin


class Decision(BaseModel):
    """Mirrors the Wahla DecisionState."""
    type: str            # loan | installment | bnpl | subscription
    monthly: float
    months: int
    hasDownPayment: bool = False


def _decision_events(d: Decision) -> list[dict]:
    """Map a Wahla decision onto Twin events.

    All four decision types are a recurring monthly commitment; a down
    payment adds a one-off expense of one installment up front.
    """
    events: list[dict] = []
    if d.hasDownPayment:
        events.append({"type": "one_off_expense", "amount": d.monthly})
    events.append({"type": "new_subscription", "monthly_amount": d.monthly,
                   "name": d.type})
    return events


@app.get("/api/accounts")
def accounts():
    out = []
    for a in DEMO_ACCOUNTS:
        twin = _TWINS.get(a["id"])
        out.append({**a, "health_score": twin.financial_health_score if twin else None})
    return out


class LiveTransaction(BaseModel):
    """One transaction as it would arrive from a live Open Banking connection.

    Fields stay loose (amount/trans_date optional-ish) on purpose: a real
    feed will contain the occasional malformed row, and rejecting the whole
    batch on one bad transaction would defeat the point of a cleaning layer.
    load_live_transactions() is where bad rows actually get dropped.
    """
    trans_id: str
    account_id: int
    trans_date: str | None = None
    amount: float | None = None
    trans_type: str
    balance: float | None = None
    operation: str | None = None
    category: str | None = None


class ConnectPayload(BaseModel):
    account_id: int
    transactions: list[LiveTransaction]
    loans: list[dict] = []
    demo: dict = {}


@app.post("/api/connect")
def connect_live_account(payload: ConnectPayload):
    """Live equivalent of build_twins.py: raw feed → cleaned → features → Twin.

    Nothing here reaches features.py/engine.py until it has passed through
    load_live_transactions()/load_live_loans() — the same cleaning contract
    the Berka batch pipeline uses in twin/data_loader.py. A malformed loan
    or transaction gets dropped by cleaning, not rejected outright: a real
    feed of thousands of records will always contain a few bad ones.
    """
    raw = [t.model_dump() for t in payload.transactions]
    tx = load_live_transactions(raw)
    if tx.empty:
        raise HTTPException(422, "no valid transactions survived cleaning")

    loans = load_live_loans(payload.loans)

    feats = build_account_features(tx, loans)
    if not feats:
        raise HTTPException(422, "not enough cleaned data to build a twin")

    twin = FinancialTwin.from_features(
        payload.account_id, feats, payload.demo, memory=build_timeline(tx, loans))
    _TWINS[payload.account_id] = twin
    return twin.to_dict()


@app.get("/api/twin/{account_id}")
def get_twin(account_id: int):
    return _twin(account_id).to_dict()


@app.post("/api/simulate/{account_id}")
def simulate(account_id: int, d: Decision):
    twin = _twin(account_id)
    sim = SimulationEngine(twin)
    result = sim._run(_decision_events(d), f"wahla_{d.type}_{d.monthly:.0f}")
    result["total_commitment"] = round(d.monthly * d.months, 2)
    return result


@app.get("/api/alternatives/{account_id}")
def alternatives(account_id: int, monthly: float, months: int):
    """For each Wahla alternative card, quantify the improvement."""
    twin = _twin(account_id)
    sim = SimulationEngine(twin)

    # reduced installment: largest amount keeping savings rate >= 10%
    headroom = max(twin.monthly_income * (twin.savings_rate - 0.10), 0)
    suggested = round(min(monthly * 0.6, headroom), -1) if headroom else round(monthly * 0.6, -1)

    base = sim.new_subscription(monthly, "current_choice")
    reduced = sim.new_subscription(suggested, "reduced") if suggested > 0 else None
    longer_monthly = round(monthly * months / 18, 2)
    longer = sim.new_subscription(longer_monthly, "longer_term")

    return {
        "current_verdict": base["verdict"],
        "reduce_payment": {
            "suggested_monthly": suggested,
            "verdict": reduced["verdict"] if reduced else None,
            "health_after": reduced["after"]["financial_health_score"] if reduced else None,
        },
        "longer_duration": {
            "months": 18,
            "monthly": longer_monthly,
            "verdict": longer["verdict"],
            "health_after": longer["after"]["financial_health_score"],
        },
        "delay": {
            "months_to_save_buffer": (
                None if twin.net_cashflow <= 0 else round(
                    max(3 * twin.monthly_expenses - twin.current_balance, 0)
                    / twin.net_cashflow)
            ),
        },
        "review_subscriptions": {
            "recurring_total": round(sum(
                p["amount"] for p in twin.recurring_payments
                if p.get("monthly")), 2),
            "count": len(twin.recurring_payments),
        },
    }


### `test_events.py` — production readiness test suite (synthetic twin, no dataset needed, runs in under a second)

In [ ]:
%%writefile test_events.py
"""Production readiness test suite — run before the live demo.

Covers:
  * every apply_event type mutates the persistent Twin (item 5)
  * every SimulationEngine scenario runs end-to-end (item 8)
  * simulations are deterministic (item 3)
  * health score is mathematically consistent after every event (item 7)
  * no consistency-check errors on any produced state (item 4/validation)

Runs on a synthetic twin — no dataset needed, executes in <1s.
"""
import copy
import json
import sys
from dataclasses import asdict

from twin.engine import FinancialTwin
from twin.features import health_score
from twin.simulation import SimulationEngine
from twin.validation import validate

FAILURES = []


def check(name: str, cond: bool, detail: str = ""):
    status = "PASS" if cond else "FAIL"
    print(f"  [{status}] {name}" + (f" — {detail}" if detail and not cond else ""))
    if not cond:
        FAILURES.append(name)


def make_twin() -> FinancialTwin:
    feats = {
        "monthly_income": 8000.0, "monthly_expenses": 5500.0,
        "spending_volatility": 0.4, "income_stability": 0.9,
        "cashflow_stability": 0.7, "monthly_loan_payment": 800.0,
        "current_balance": 24000.0, "detected_salary": 8000.0,
        "weekend_spending_ratio": 0.3, "cash_usage_ratio": 0.2,
        "overdraft_months": 0, "months_of_history": 36,
        "debt_ratio": 0.1, "savings_rate": 0.3,   # will be re-derived
        "emergency_fund_months": 4.4, "net_cashflow": 2500.0,
        "financial_health_score": 0.0,
        "recurring_payments": [],
    }
    return FinancialTwin.from_features(999, feats)


EVENTS = [
    {"type": "transaction", "amount": -450, "category": "household"},
    {"type": "salary_change", "new_salary": 9000},
    {"type": "new_loan", "principal": 30000, "duration_months": 48,
     "annual_rate": 0.05, "disbursed_to_account": 30000},
    {"type": "loan_payoff", "monthly_payment": 800},
    {"type": "new_subscription", "monthly_amount": 60, "name": "streaming"},
    {"type": "cancel_subscription", "monthly_amount": 60},
    {"type": "rent_change", "delta": 300},
    {"type": "one_off_expense", "amount": 2000},
    {"type": "investment", "monthly_amount": 500},
]

SIMULATIONS = [
    ("buy_item", {"price": 3000}),
    ("buy_with_bnpl", {"price": 3000, "installments": 4}),
    ("take_loan", {"principal": 20000, "duration_months": 36}),
    ("buy_car", {"price": 25000}),
    ("new_subscription", {"monthly_amount": 50}),
    ("rent_increase", {"delta": 400}),
    ("salary_change", {"new_salary": 10000}),
    ("medical_emergency", {"cost": 5000}),
    ("vacation", {"cost": 4000}),
    ("invest_monthly", {"monthly_amount": 800}),
    ("payoff_debt", {"monthly_payment": 800}),
]


def test_events():
    print("\n== every event type updates the persistent Twin ==")
    for e in EVENTS:
        twin = make_twin()
        before = twin.snapshot()
        report = twin.apply_event(dict(e))
        after = twin.snapshot()
        check(f"event {e['type']}: state changed", before != after)
        check(f"event {e['type']}: change report non-empty", len(report) > 0)
        check(f"event {e['type']}: appended to memory",
              twin.memory and twin.memory[-1]["type"] == e["type"])
        # health score mathematically consistent with current state
        expected = health_score(asdict(twin))
        check(f"event {e['type']}: health score consistent",
              abs(twin.financial_health_score - expected) < 0.05,
              f"stored {twin.financial_health_score} vs recomputed {expected}")
        errors = [w for w in validate(asdict(twin)) if w["severity"] == "error"]
        check(f"event {e['type']}: no validation errors",
              not errors, str(errors))


def test_simulations():
    print("\n== every simulation runs end-to-end, deterministically ==")
    twin = make_twin()
    sim = SimulationEngine(twin)
    for method, args in SIMULATIONS:
        r1 = getattr(sim, method)(**args)
        r2 = getattr(sim, method)(**args)
        check(f"sim {method}: runs", "after" in r1 and "verdict" in r1)
        check(f"sim {method}: deterministic", r1["after"] == r2["after"],
              "two identical runs differ")
        check(f"sim {method}: exposes transition",
              r1["transition"]["from_state"] == r1["before"]
              and r1["transition"]["to_state"] == r1["after"])
        check(f"sim {method}: twin_diff present", isinstance(r1["twin_diff"], list))
        check(f"sim {method}: explanation references diff",
              bool(r1["explanation"]))
        check(f"sim {method}: real twin untouched",
              twin.snapshot() == r1["before"])
        errors = [w for w in r1["validation"] if w["severity"] == "error"]
        check(f"sim {method}: no validation errors", not errors, str(errors))


def test_compare():
    print("\n== multi-scenario comparison ==")
    sim = SimulationEngine(make_twin())
    ranked = sim.compare([
        {"method": "buy_car", "args": {"price": 25000}},
        {"method": "buy_item", "args": {"price": 25000, "label": "cash"}},
        {"method": "invest_monthly", "args": {"monthly_amount": 400}},
    ])
    scores = [r["after"]["financial_health_score"] for r in ranked]
    check("compare: ranked by health desc", scores == sorted(scores, reverse=True))


if __name__ == "__main__":
    test_events()
    test_simulations()
    test_compare()
    print(f"\n{'ALL TESTS PASSED' if not FAILURES else f'{len(FAILURES)} FAILURES: {FAILURES}'}")
    sys.exit(1 if FAILURES else 0)


### `make_demo.py` — two real, contrasting personas: same car, same question, opposite verdicts

In [ ]:
%%writefile make_demo.py
"""Build the live-demo payload: two real contrasting personas + simulations.

Account 2  — goal-oriented planner who repaid a real loan successfully.
Account 19 — impulse spender whose real loan defaulted.

Same engine, same question ("can I afford a 20k car?"), opposite answers —
that is the demo. Output: data/processed/demo_story.json (frontend-ready).

Run: python3 make_demo.py   (rebuilds the two twins from raw data, so the
full lineage raw → twin → simulation is reproducible live)
"""
import json
import sys

from twin import config
from twin.data_loader import load_all
from twin.features import build_account_features
from twin.memory import build_timeline
from twin.engine import FinancialTwin
from twin.simulation import SimulationEngine

# Two real clients with nearly IDENTICAL incomes (~1,400-1,600/month) and
# opposite financial lives — same car, same question, opposite verdicts.
DEMO_ACCOUNTS = {
    21: "The Disciplined Saver — saves 56% of income, 8.5 months of runway",
    19: "The Impulse Spender — real loan ended in default, overdraft history",
}
CAR_PRICE = 6000


def build_twin(data: dict, acct_id: int) -> FinancialTwin:
    tx = data["transactions"]
    acct_tx = tx[tx["account_id"] == acct_id]
    acct_loans = data["loans"][data["loans"]["account_id"] == acct_id]
    feats = build_account_features(acct_tx, acct_loans)
    timeline = build_timeline(acct_tx, acct_loans)
    return FinancialTwin.from_features(acct_id, feats, memory=timeline)


def main():
    print("loading data...", file=sys.stderr)
    data = load_all()

    story = {}
    for acct_id, headline in DEMO_ACCOUNTS.items():
        twin = build_twin(data, acct_id)
        sim = SimulationEngine(twin)
        story[str(acct_id)] = {
            "headline": headline,
            "twin": twin.to_dict(),
            "what_if_car": sim.buy_car(CAR_PRICE),
            "what_if_compare": sim.compare([
                {"method": "buy_car", "args": {"price": CAR_PRICE}},
                {"method": "buy_with_bnpl", "args": {"price": CAR_PRICE,
                                                     "installments": 12}},
                {"method": "invest_monthly", "args": {"monthly_amount": 350}},
            ]),
        }
        r = story[str(acct_id)]["what_if_car"]
        print(f"\naccount {acct_id} ({twin.financial_personality}, "
              f"health {twin.financial_health_score}):")
        print(f"  car verdict: {r['verdict']}")
        print(f"  {r['explanation']}")

    out = config.OUTPUT_DIR / "demo_story.json"
    with open(out, "w") as f:
        json.dump(story, f, indent=1)
    print(f"\ndemo payload → {out}")


if __name__ == "__main__":
    main()


## 3. Now run it — every call below executes the code you just read (no subprocess)

Load + clean the raw Berka export via the `data_loader.py` you just wrote.

In [ ]:
import sys
sys.path.insert(0, ".")

from twin.data_loader import load_all

data = load_all()
print(f"transactions after cleaning: {len(data['transactions']):,}")
print(f"loans after cleaning: {len(data['loans']):,}")


Build every Twin (`build_twins.build_all_twins`) and export `data/processed/twins.json` — this is what `api.py` loads on startup.

In [ ]:
import build_twins

twins = build_twins.build_all_twins(data)
build_twins.export(twins)
print(f"built {len(twins)} twins")


Demo simulations for account 19 — buy a car, then compare car-loan vs cash vs invest-the-installment.

In [ ]:
build_twins.demo_simulations(twins[19])


Judge mode: full lineage for account 19 — raw rows → features → state → event → new state → why.

In [ ]:
build_twins.inspect_account(data, twins[19], 19)


## 3b. Two real, contrasting personas — same car, same question, opposite verdicts

Account 21 saves 56% of income with 8.5 months of runway; account 19's real
loan (in the actual Berka data) ended in default. Same engine, same 6,000
car, asked of both.

In [ ]:
import make_demo
from twin.simulation import SimulationEngine

for acct_id, headline in make_demo.DEMO_ACCOUNTS.items():
    twin = make_demo.build_twin(data, acct_id)
    sim = SimulationEngine(twin)
    r = sim.buy_car(make_demo.CAR_PRICE)
    print(f"\naccount {acct_id} — {headline}")
    print(f"  personality: {twin.financial_personality}, health: {twin.financial_health_score}")
    print(f"  car verdict: {r['verdict']}")
    print(f"  {r['explanation']}")


## 4. Architecture isolation proof

In [ ]:
import check_architecture
check_architecture.run_check()


## 4b. Production readiness test suite — every event type and every simulation, checked for determinism and consistency

In [ ]:
import test_events

test_events.FAILURES.clear()
test_events.test_events()
test_events.test_simulations()
test_events.test_compare()
print(f"\n{'ALL TESTS PASSED' if not test_events.FAILURES else f'{len(test_events.FAILURES)} FAILURES: ' + str(test_events.FAILURES)}")


## 5. Live cleaning boundary — the connect-time equivalent of the batch pipeline

Deliberately dirty sample: a duplicate transaction, one with an unparseable
date, one missing its amount, and a loan missing its `loan_id`. Watch the row
count drop from what goes in to what survives `load_live_transactions()` /
`load_live_loans()` — the same functions defined in the `data_loader.py` cell
above.

In [ ]:
from twin.data_loader import load_live_transactions, load_live_loans

dirty_transactions = [
    {"trans_id": "a1", "account_id": 99001, "trans_date": "2024-02-01T09:00:00",
     "amount": 6000, "balance": 6000, "trans_type": "credit", "category": "salary"},
    {"trans_id": "a2", "account_id": 99001, "trans_date": "2024-02-05",
     "amount": -800, "balance": 5200, "trans_type": "debit", "category": "household"},
    {"trans_id": "a2", "account_id": 99001, "trans_date": "2024-02-05",
     "amount": -800, "balance": 5200, "trans_type": "debit", "category": "household"},
    {"trans_id": "a3", "account_id": 99001, "trans_date": "not-a-date",
     "amount": 50, "trans_type": "debit"},
    {"trans_id": "a4", "account_id": 99001, "trans_date": "2024-02-10",
     "amount": None, "trans_type": "debit"},
]
tx = load_live_transactions(dirty_transactions)
print(f"transactions: {len(dirty_transactions)} in -> {len(tx)} survived cleaning")
print(tx[["trans_id", "trans_date", "amount", "trans_type", "category"]])

dirty_loans = [
    {"loan_id": 501, "account_id": 99001, "granted_date": "2023-05-01",
     "amount": 15000, "duration": 24, "payments": 700, "status": "running_ok"},
    {"loan_id": None, "account_id": 99001, "granted_date": "2023-06-01",
     "amount": 5000, "duration": 6, "payments": 900, "status": "running_ok"},
]
loans = load_live_loans(dirty_loans)
print(f"\nloans: {len(dirty_loans)} in -> {len(loans)} survived cleaning")
print(loans[["loan_id", "account_id", "status"]])


## 6. Run the live API inside this notebook

`api.py` (written above) is a normal FastAPI app — launched here on
`127.0.0.1:8000` in a background thread so the cells below can call it exactly
like the real frontend would, over real HTTP.

In [ ]:
import threading, time
import nest_asyncio
import uvicorn

nest_asyncio.apply()

import api as wahla_api

def _run_server():
    uvicorn.run(wahla_api.app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=_run_server, daemon=True).start()
time.sleep(2)

import requests
BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/api/accounts").json())


### 6a. Connect a live account — same dirty sample, this time over real HTTP through `/api/connect`

In [ ]:
connect_payload = {
    "account_id": 99001,
    "transactions": dirty_transactions,
    "loans": dirty_loans,
}
r = requests.post(f"{BASE}/api/connect", json=connect_payload)
print("status:", r.status_code)
twin_state = r.json()
print("financial_health_score:", twin_state.get("financial_health_score"))
print("current_balance:", twin_state.get("current_balance"))


### 6b. Simulate a decision on a real demo account

In [ ]:
decision = {"type": "loan", "monthly": 900, "months": 24, "hasDownPayment": False}
r = requests.post(f"{BASE}/api/simulate/19", json=decision)
print(r.status_code)
r.json()


In [ ]:
r = requests.get(f"{BASE}/api/alternatives/19", params={"monthly": 900, "months": 24})
print(r.status_code)
r.json()


## Where to read more

- `README.md` — architecture overview and pipeline diagram
- `ARCHITECTURE.md` — full design rationale
- `HACKATHON.md` — the 90-second "why this is a real Digital Twin, not a dashboard" answer
